# Exact Numeric Solution of the Moran Process

`src/moran_process/numeric` (Rust) replaces Monte Carlo sampling with the exact answer. On a
graph of N nodes a state is the set of nodes holding the mutant, so there are 2^N states, and
both the fixation probability and the expected absorption time obey one linear equation per
state. The solver runs Gauss-Seidel with over-relaxation (omega = 1.5) over the whole state
vector, visiting states in Gray-code order so that consecutive states differ in one node and
the transition weights are patched in O(degree) instead of being rebuilt. The vector is cut
into 256 divisions that threads pull from a shared schedule. One solve gives the answer for
every starting node at once.

**The cost is the state vector**, 8 bytes x 2^N. Measured on binary trees, r = 1.1, 8 cores:

| N | 20 | 22 | 24 |
|---|---|---|---|
| wall time | 5.6 s | 28 s | 143 s |

a factor of ~2.25 per added node. N = 21 is 16 MB and seconds, N = 31 is 17 GB and hours. The
solver takes N <= 63, and N = 8 did not terminate in a test, so use N >= 9.

**How to run.** Run section 0, then either section on its own. A section's first cell checks
every result it needs: loaded if its file exists, shown with its LSF state if its job is queued
or running, submitted otherwise. Nothing waits. Unless everything is ready the cell raises, so
Run All stops there instead of drawing half the results: run it again later (`bjobs -w` shows
progress). Every job writes
`simulation_data/numeric_solutions/results/<graph>_r<r>_<prob|time>_c<cores>.txt`, and a file
that exists is never recomputed, so reopening the notebook only reads files.

---

## 0. Setup (run first)

`collect` checks a set of results: it loads the ready ones, reports the ones LSF is still
working on, submits the missing ones, and raises unless all are ready. Memory is sized from N,
since the solver holds 2^N doubles.

In [ ]:
%load_ext autoreload
%autoreload 2
%cd /home/labs/pilpel/matanyaw/moran-process

import re
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import linalg

from moran_process import PopulationGraph
from moran_process.analysis.analysis_utils.theory import analytic_moran_fc_fixation_prob

from enum import StrEnum

class Action(StrEnum):
    PROB = "prob"
    TIME = "time"

class Queue(StrEnum):
    GSLA = "gsla-cpu"
    SHORT = "short"
    MEDIUM = "medium"
    Long = "long"

CLI = Path("src/moran_process/numeric/target/release/numeric_cli").resolve()
OUT = Path("simulation_data/numeric_solutions").resolve()
for d in ("graphs", "results", "logs"):
    (OUT / d).mkdir(parents=True, exist_ok=True)


In [ ]:
def job_name(g, r, action, cores):
    """One name for a job's LSF job, result file and log."""
    return f"{g.name}_r{r}_{action}_c{cores}"


def submit(g, r, action, cores, queue):
    """Queue one CLI run of `action` on `g` at `r`, with memory sized from N."""
    name = job_name(g, r, action, cores)
    out = OUT / "results" / f"{name}.txt"
    graph_file = OUT / "graphs" / f"{g.name}.txt"
    g.to_simulation_struct().save(graph_file)
    mem_mb = 2**g.n_nodes * 8 / 2**20 * 1.2 + 512  # the solver holds 2**N doubles, plus headroom
    # The script goes to bsub on stdin: the WEXAC job starter mangles $variables and
    # quotes in a command given on the bsub line.
    subprocess.run(
        ["bsub", "-q", queue, "-J", name, "-n", str(cores),
         "-R", f"span[hosts=1] rusage[mem={int(mem_mb / cores) + 1}]",  # LSF reserves per core
         "-oo", str(OUT / "logs" / f"{name}.log")],
        input=f"{CLI} {action} --file {graph_file} {r} > {out}.tmp && mv {out}.tmp {out}\n",
        text=True, check=True, capture_output=True,
    )


def collect(jobs, queue=Queue.SHORT):
    """Load the results of `jobs` ({key: (graph, r, action, cores)}), never waiting.

    Every job is ready (its result file exists, and is loaded), in LSF (shown with
    its state, PEND, RUN, ..., and when it was sent), failed (its log says so;
    delete the log to rerun), or submitted now. Raises unless every job is ready,
    so Run All stops here: run the cell again later.
    """
    results, not_ready = {}, []
    for key, (g, r, action, cores) in jobs.items():
        name = job_name(g, r, action, cores)
        out, log = OUT / "results" / f"{name}.txt", OUT / "logs" / f"{name}.log"
        if out.exists():
            results[key] = np.loadtxt(out)
            continue
        lsf = subprocess.run(["bjobs", "-noheader", "-o", "stat submit_time", "-J", name],
                             capture_output=True, text=True).stdout
        if lsf:
            state, sent = lsf.splitlines()[0].split(maxsplit=1)  # e.g. "RUN", "Sep 15 10:50"
        elif log.exists() and "Exited with exit code" in log.read_text():
            state, sent = f"FAILED, delete {log} to rerun", ""
        else:
            submit(g, r, action, cores, queue)
            state, sent = "submitted", pd.Timestamp.now().strftime("%b %d %H:%M")
        not_ready.append(f"  {name:36} {state:9} {'sent ' + sent if sent else ''}")

    print(f"{len(results)}/{len(jobs)} ready")
    if not_ready:
        print("\n".join(not_ready))
        raise RuntimeError(f"{len(not_ready)} result(s) not ready: run this cell again later")
    return results


def run_hours(name):
    """Wall time of a finished job, from its LSF log."""
    log = (OUT / "logs" / f"{name}.log").read_text()
    return int(re.search(r"Run time :\s+(\d+) sec", log).group(1)) / 3600

In [ ]:
MAPS = {
    Action.PROB: dict(cmap="RdYlGn", label="fixation probability"),
    Action.TIME: dict(cmap="viridis", label="mean steps to absorption"),
}


def draw_panel(g, v, kind, r, ax, node_size=350, title=None):
    """Draw `g` on `ax` with nodes colored by `v`, a `kind` result (scale rule in section 1)."""
    if kind == "prob":
        center = float(analytic_moran_fc_fixation_prob(g.n_nodes, r))
        floor = 0.5 * center
    else:
        center = float(v.mean())
        floor = 0.1 * center
    g.draw_colored_graph(
        v, **MAPS[kind], center=center, vmin_floor=0.0,
        half_width=max(float(np.abs(v - center).max()), floor),
        node_size=node_size, ax=ax, title=title,
    )


def draw_maps(graphs, results, kind, r, ncols=4, node_size=350):
    """One panel per graph, nodes colored by `results[g.name, kind]`."""
    ncols = min(ncols, len(graphs))
    nrows = -(-len(graphs) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 5.0 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, g in zip(axes, graphs):
        draw_panel(g, results[g.name, kind], kind, r, ax, node_size,
                   title=f"{g.name}  (N={g.n_nodes})")
    for ax in axes[len(graphs):]:
        ax.axis("off")
    fig.suptitle(f"Exact {MAPS[kind]['label']} by starting node  (r = {r})", fontsize=16, y=1.0)
    plt.tight_layout()
    plt.show()


def draw_pairs(graphs, results, r, node_size=350):
    """One figure per graph: fixation probability (left) and time to absorption (right)."""
    for g in graphs:
        fig, axes = plt.subplots(1, 2, figsize=(5.5 * 2, 5.0))
        for ax, kind in zip(axes, MAPS):
            draw_panel(g, results[g.name, kind], kind, r, ax, node_size,
                       title=MAPS[kind]["label"])
        fig.suptitle(f"{g.name}  (N={g.n_nodes}, r = {r})", fontsize=14)
        plt.tight_layout()
        plt.show()

---

## 1. Per-node maps

The zoo of `multicolor_simulation.ipynb`, minus what the solver cannot handle yet:

* the 66-node fish is swapped for an 18-node one, since 2^66 states would not fit anywhere;
* `directed_mammalian` and `directed_star` are out: their leaves have out-degree 0, which the
  CLI's text format rejects ("bad file contents"), and the library call returns probabilities
  above 1 and negative times on them;
* `directed_cycle` is out: it returns in 0.2 s (the undirected cycle takes 14 s) with
  probabilities from -9.7 to 7.3. Most likely the over-relaxed iteration diverges on it, and
  the stopping rule, which ends the run once the changes stop being positive, takes the
  oscillation for convergence.

`directed_avian` stays, because it matches theory: to 1e-15 at `rod_length=2`, to 1e-6 at
`rod_length=3`. Treat any other directed graph as unverified until it passes that check. Jobs
take seconds per graph up to N = 21; a 31-node graph takes about a day on 4 cores (section 2
solves the 31-node lung, and shares these result files).

In [ ]:
R = 1.1

GRAPHS = [
    # --- undirected ---
    PopulationGraph.mammalian_lung_graph(branching_factor=2, depth=3),
    PopulationGraph.avian_graph(n_rods=6, rod_length=3),
    PopulationGraph.fish_graph(n_rods=2, rod_length=2, fillaments=3),
    PopulationGraph.complete_graph(21),
    PopulationGraph.cycle_graph(21),
    PopulationGraph.star_graph(21),
    PopulationGraph.line_graph(21),
    PopulationGraph.grid_graph(4, 5),
    # --- directed: offspring flow along edge direction only ---
    PopulationGraph.avian_graph(n_rods=6, rod_length=3, directed=True),
]

maps = collect({(g.name, act): (g, R, act, 4) for g in GRAPHS for act in Action})

### 1a. The maps

At r = 1 a single mutant at v fixes exactly when v's lineage wins the multicolor race: a
neutral mutant is one lineage among N, colored differently. So the probability map is the
multicolor win-fraction map with the Monte Carlo noise removed. The time map is a **different
quantity**: unconditional steps until the two-type process absorbs, not the multicolor
consensus time conditional on which node won.

Same scale rule as `multicolor_simulation.ipynb`: center each panel on the value a graph
without positional structure would show, and widen to the larger of the largest deviation and
a floor. Probability centers on the well-mixed value (1/N at r = 1) with a floor of half of
it, as there. Time centers on the graph's own mean; the multicolor floor was a noise level
built from the SEM, and exact values have none, so here the floor is an effect size: 10% of
the mean.

In [ ]:
draw_pairs(GRAPHS, maps, R)

---

## 2. Cores vs wall time on the 31-node lung

Independent of section 1: run section 0, then this. Both quantities are solved at every core
count, each as its own job, so each time is one clean solve, read from the job's LSF log
(`Run time`, 1 s resolution).

`mammalian_b2_d4` has 31 nodes: 2^31 states, ~17 GB per job whatever the core count. Expect
roughly 11 h per solve on 8 cores and about a day on 4, hence the `medium` queue (72 h). Run
the first cell to submit, then again whenever you want to check: it lists each job's state
and raises until all 8 are done. `medium` is preemptable, and a preempted job restarts from
scratch, so a point that looks off may have been preempted or suspended: check its log.
Every core count solves the same system, so the values must agree across them, which the
second cell prints as a check.

In [ ]:
LUNG = PopulationGraph.mammalian_lung_graph(branching_factor=2, depth=4)  # N = 31
R_LUNG = 1.1
CORES = [4, 8, 16, 32]

lung = collect({(act, c): (LUNG, R_LUNG, act, c) for act in Action for c in CORES},
               queue=Queue.MEDIUM)

In [ ]:
hours = pd.DataFrame({act: {c: run_hours(job_name(LUNG, R_LUNG, act, c)) for c in CORES}
                      for act in Action})
hours.index.name = "cores"
speedup = hours.iloc[0] / hours  # relative to the fewest cores
display(pd.concat({"hours": hours, "speedup": speedup}, axis=1).round(2))
for act in Action:
    ref = lung[act, CORES[0]]
    print(f"{act}: max |difference| across core counts = "
          f"{max(np.abs(lung[act, c] - ref).max() for c in CORES):.1e}")

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(hours.index, hours.index / hours.index[0], "k--", lw=1, label="ideal")
for act in Action:
    ax.plot(speedup.index, speedup[act], "o-", label=MAPS[act]["label"])
ax.set_xticks(hours.index)
ax.set(xlabel="cores requested (bsub -n)", ylabel=f"speedup over {hours.index[0]} cores",
       title=f"{LUNG.name} (N = {LUNG.n_nodes}), r = {R_LUNG}")
ax.legend()
plt.show()

draw_pairs([LUNG], {(LUNG.name, act): lung[act, CORES[0]] for act in Action}, R_LUNG)